# DriveDE - 3D Roundabout Render (Blender on Colab)

Renders the procedural low-poly roundabout explainer (German rules: yield to ring traffic,
enter without signaling, signal right to exit) as 480 frames / 16 s at 720x1280.

**How to run:** Runtime -> Change runtime type -> **T4 GPU**, then Runtime -> **Run all**.
Takes ~25-35 min total (Blender download ~3 min, render ~20-30 min on the T4).
The last cell downloads `roundabout-3d.mp4`. Send it to Claude Code for compositing.


In [ ]:
# @title Step 1: Download Blender + the scene script
import os
if not os.path.exists('/content/blender'):
    !wget -q https://download.blender.org/release/Blender4.2/blender-4.2.3-linux-x64.tar.xz -O /content/blender.tar.xz
    !tar -xf /content/blender.tar.xz -C /content
    !mv /content/blender-4.2.3-linux-x64 /content/blender
    !apt-get -qq install -y libxi6 libxrender1 libxkbcommon0 libsm6 > /dev/null
!wget -q -O /content/roundabout.py https://raw.githubusercontent.com/abhijit5721/DriveDE/staging/content/blender/roundabout.py
!/content/blender/blender --version | head -1
print('setup done')


In [ ]:
# @title Step 2a: 1-minute smoke test (10 frames) - proves GPU + script + encoder work
!mkdir -p /content/smoke
!/content/blender/blender --background --python /content/roundabout.py -- --animate --start 200 --end 209 --out /content/smoke --samples 16 2>&1 | tail -2
import glob
n = len(glob.glob('/content/smoke/*.png'))
print(f'{n}/10 smoke frames OK')
assert n == 10, 'smoke test failed - fix before the long render'


In [ ]:
# @title Step 2b: Full render (resumable - re-running skips finished frames)
import glob, subprocess, os
os.makedirs('/content/frames', exist_ok=True)
done = sorted(int(f.split('f_')[1][:4]) for f in glob.glob('/content/frames/f_*.png'))
start = (done[-1] + 1) if done else 1
print(f'{len(done)} frames already done, starting at frame {start}')
if start <= 480:
    subprocess.run(['/content/blender/blender', '--background', '--python', '/content/roundabout.py',
                    '--', '--animate', '--start', str(start), '--out', '/content/frames', '--samples', '32'])
n = len(glob.glob('/content/frames/f_*.png'))
print(f'{n}/480 frames rendered')
assert n >= 480, 're-run this cell to resume from where it stopped'


In [ ]:
# @title Step 3: Encode + download roundabout-3d.mp4
!ffmpeg -y -loglevel error -framerate 30 -i /content/frames/f_%04d.png -c:v libx264 -pix_fmt yuv420p -crf 16 /content/roundabout-3d.mp4
import os
print(os.path.getsize('/content/roundabout-3d.mp4'), 'bytes')
from google.colab import files
files.download('/content/roundabout-3d.mp4')
